# RRF 하이브리드 검색

이 노트북은 빈 Python 3.12 커널에서 순서대로 실행합니다.


## 실행 환경 준비

05절과 같은 두 검색기의 순위를 상호 순위 융합(reciprocal rank fusion, RRF)으로 합칩니다. API 예상 비용은 미화 0.01달러 미만입니다.

### 서로 다른 점수를 순위로 합치기

dense 거리와 BM25 원점수는 그대로 더할 수 없어 RRF는 각 목록의 순위 `r`를 `weight / (c + r)`로 바꿔 더합니다. 두 목록 모두 상위에 둔 문서는 두 항을 받습니다. sparse 가중치를 높이는 것은 BM25 원점수가 아니라 sparse 순위의 기여를 키우는 정책입니다.


In [1]:
%pip install -q "numpy==2.5.3" "kiwipiepy==0.23.2" "rank-bm25==0.2.2" "python-dotenv==1.2.3" "langchain-openai==1.6.1"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, os
from pathlib import Path
import numpy as np
from dotenv import find_dotenv, load_dotenv
from langchain_openai import OpenAIEmbeddings

# 현재 폴더부터 상위 폴더에서 .env를 찾고 키의 존재만 확인합니다.
env_file = find_dotenv(usecwd=True)
assert env_file, "프로젝트 루트에 .env를 준비하세요."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY를 확인하세요."
course_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "labs/part3/ch12-retrieval-tuning/data").is_dir()), None)
assert course_root is not None, "Ch 12 실습 데이터 폴더를 찾지 못했습니다."
chapter_dir = course_root / "labs/part3/ch12-retrieval-tuning"
data_dir = chapter_dir / "data"
docs = json.loads((data_dir / "core-documents.json").read_text())
queries = json.loads((data_dir / "core-queries.json").read_text())
embedder = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=256)

In [3]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

In [4]:
texts = [f"{doc['title']} {doc['text']}" for doc in docs]
doc_vectors = np.array(embedder.embed_documents(texts))
query_vectors = np.array([embedder.embed_query(q["question"]) for q in queries])
kiwi = Kiwi(); keep = ("N", "V", "VA", "MAG", "SN", "SL")
def terms(text):
    return [token.form for token in kiwi.tokenize(text) if token.tag.startswith(keep)]
bm25 = BM25Okapi([terms(text) for text in texts])
def dense_rank(vector, k=3):
    scores = doc_vectors @ vector / (np.linalg.norm(doc_vectors, axis=1) * np.linalg.norm(vector))
    return [docs[i]["doc_id"] for i in np.argsort(-scores)[:k]]
def sparse_rank(question, k=3):
    scores = bm25.get_scores(terms(question))
    order = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)[:k]
    return [docs[i]["doc_id"] for i in order]

## 기본 실행

RRF는 서로 다른 원점수를 더하지 않고 각 검색기의 순위만 사용합니다.


In [5]:
def rrf(rankings, weights=None, rank_constant=60, k=3):
    weights = weights or [1.0] * len(rankings)
    scores = {}
    for ranking, weight in zip(rankings, weights):
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + weight / (rank_constant + rank)
    return sorted(scores, key=scores.get, reverse=True)[:k], scores

query = queries[2]
dense = dense_rank(query_vectors[2], 6)
sparse = sparse_rank(query["question"], 6)
fused, scores = rrf([dense, sparse])
print("dense:", dense); print("sparse:", sparse); print("RRF:", fused)

dense: ['LIB-003', 'ACC-002', 'LIB-004', 'ACC-004', 'ACC-003', 'LIB-001']
sparse: ['LIB-003', 'ACC-003', 'LIB-004', 'ACC-002', 'LIB-001', 'LIB-002']
RRF: ['LIB-003', 'ACC-002', 'LIB-004']


## 중간 결과 관찰

한쪽에만 있는 문서도 점수를 받고, 두 순위 모두에서 상위에 있는 문서는 두 항을 더합니다.


In [6]:
for doc_id in fused:
    dense_rank_no = dense.index(doc_id) + 1 if doc_id in dense else None
    sparse_rank_no = sparse.index(doc_id) + 1 if doc_id in sparse else None
    print(doc_id, "dense 순위", dense_rank_no, "sparse 순위", sparse_rank_no,
          "RRF 점수", round(scores[doc_id], 5))

LIB-003 dense 순위 1 sparse 순위 1 RRF 점수 0.03279
ACC-002 dense 순위 2 sparse 순위 4 RRF 점수 0.03175
LIB-004 dense 순위 3 sparse 순위 3 RRF 점수 0.03175


## 값 하나를 바꿔 비교

나머지 설정은 유지하고 sparse 가중치만 1.0에서 1.5로 바꿉니다.


In [7]:
equal, _ = rrf([dense, sparse], [1.0, 1.0])
sparse_up, _ = rrf([dense, sparse], [1.0, 1.5])
print("동일 가중치:", equal)
print("sparse 1.5:", sparse_up)

동일 가중치: ['LIB-003', 'ACC-002', 'LIB-004']
sparse 1.5: ['LIB-003', 'LIB-004', 'ACC-003']


## 반복 실험과 결과 해석

8개 질문에서 dense, sparse, 동일 가중치 RRF의 Recall@3를 비교합니다.


In [8]:
rows = []
for query, vector in zip(queries, query_vectors):
    dense = dense_rank(vector, 6); sparse = sparse_rank(query["question"], 6)
    fused, _ = rrf([dense, sparse])
    gold = query["gold_doc_id"]
    rows.append({"qid": query["qid"], "gold": gold, "dense": dense[:3],
                 "sparse": sparse[:3], "rrf": fused,
                 "hits": {"dense": gold in dense[:3], "sparse": gold in sparse[:3], "rrf": gold in fused}})
for name in ["dense", "sparse", "rrf"]:
    print(name, "Recall@3:", sum(row["hits"][name] for row in rows), "/", len(rows))
print("기록할 결과 행:", len(rows))

dense Recall@3: 7 / 8
sparse Recall@3: 8 / 8
rrf Recall@3: 7 / 8
기록할 결과 행: 8


<h3>실행 결과 보존</h3><p>기존 정식 결과를 해시 이름으로 보존하고, 이번 결과를 실행 시각이 포함된 파일에 먼저 저장한 뒤 최신 결과 파일을 갱신합니다.</p>

In [9]:
import hashlib
from datetime import UTC, datetime
payload = json.dumps(rows, ensure_ascii=False, indent=2)
runs = chapter_dir / "runs"; history = runs / "core-history"
history.mkdir(parents=True, exist_ok=True)
pointer = runs / "core-06-rrf.json"
if pointer.exists():
    old = pointer.read_bytes()
    backup = history / f"before-{hashlib.sha256(old).hexdigest()[:12]}-{pointer.name}"
    if not backup.exists(): backup.write_bytes(old)
run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%f")
archive = history / f"{run_id}-{pointer.name}"
archive.write_text(payload, encoding="utf-8")
pointer.write_text(payload, encoding="utf-8")
print("실행별 보존 로그:", archive.relative_to(course_root))

실행별 보존 로그: labs/part3/ch12-retrieval-tuning/runs/core-history/20260909T174519370916-core-06-rrf.json
